# 🐍 Snake Detector — TFLite INT8 Export (Bulletproof Fix)
**Run Cell 1 → Restart Session → Run all remaining cells**

This version finds your true `data.yaml` from your previous training so the INT8 calibration doesn't crash.

## Cell 1: Install & Restart Session

In [ ]:
!pip install -U ultralytics
print("\n✅ Done — NOW: Runtime → Restart session, then run Cell 2 onwards")

## Cell 2: Find Model & Dataset Configuration

In [ ]:
from ultralytics import YOLO
import os, glob

HOME = os.getcwd()
best_pt = None

# ─── 1. Find best.pt ───
for p in [f"{HOME}/best.pt", "/content/best.pt"]:
    if os.path.exists(p): best_pt = p
if not best_pt:
    found = glob.glob("/content/**/best.pt", recursive=True)
    if found: best_pt = found[0]
if not best_pt:
    from google.colab import files
    print("Upload best.pt:")
    up = files.upload()
    for f in up: 
        if f.endswith('.pt'): best_pt = os.path.join(HOME, f)
if not best_pt: raise FileNotFoundError("No best.pt found!")
print(f"✅ Model: {best_pt}")

# ─── 2. Find data.yaml for INT8 Calibration ───
data_yaml = None
model = YOLO(best_pt)

# First try: The exact path from your previous log
exact_path = "/content/snake_yolo_dataset/data.yaml"
if os.path.exists(exact_path):
    data_yaml = exact_path

# Second try: extract from the model's training arguments
if not data_yaml and hasattr(model, 'task_map'):
    try: 
        saved_data = model.ckpt.get('train_args', {}).get('data')
        if saved_data and os.path.exists(saved_data):
            data_yaml = saved_data
    except: pass

# Third try: deep search
if not data_yaml:
    yamls = glob.glob("/content/**/*.yaml", recursive=True)
    for y in yamls:
        if 'data.yaml' in y or 'dataset.yaml' in y:
            data_yaml = y
            break

# Ultimate fallback: Download coco8 (better than crashing)
if not data_yaml:
    print("⚠️ No true data.yaml found! Falling back to coco8.yaml for calibration.")
    data_yaml = "coco8.yaml"

print(f"✅ INT8 Calibration Data: {data_yaml}")

## Cell 3: Export INT8

In [ ]:
print("\n🔄 Exporting INT8 TFLite (this takes ~3-5 minutes)...")
try:
    tflite_int8_path = model.export(
        format="tflite",
        imgsz=320,
        int8=True,       
        data=data_yaml,
    )
    print(f"\n🎉 SUCCESS! INT8 model created: {tflite_int8_path}")
except Exception as e:
    print(f"\n❌ CRITICAL FATAL ERROR: {e}")

## Cell 4: Download

In [ ]:
import shutil, glob
from google.colab import files

deploy_dir = "snake_model_int8_final"
if os.path.exists(deploy_dir): shutil.rmtree(deploy_dir)
os.makedirs(deploy_dir)

tflite_files = glob.glob("/content/**/*.tflite", recursive=True)
for src in tflite_files:
    if "int8" in src.lower() or "quant" in src.lower() or "saved_model" in src.lower():
        shutil.copy2(src, os.path.join(deploy_dir, os.path.basename(src)))

shutil.make_archive(deploy_dir, "zip", deploy_dir)
files.download(f"{deploy_dir}.zip")
print(f"\n📦 Downloading {deploy_dir}.zip")
print("   Move ALL included files to c:\\IOT PROJECTS\\BIO_MIMIC\\models\\")